# V2 CUDA Test Notebook

Notebook này giữ cùng bố cục kiểm tra với `test_v1.ipynb`, bổ sung Version 2:
**S4 matching dùng shared-memory descriptor tiling**. S1–S3, S5–S6 dùng lại V1.

Các kiểm tra gồm:
- V0 end-to-end panorama làm reference.
- S1–S2 CUDA parity test (kế thừa V1).
- S3–S4 CUDA parity test cho V0/V1/V2, gồm biên tile và tie-break.
- S5 CUDA sampling/RANSAC/refinement parity test (kế thừa V1).
- S6 CUDA verification test (kế thừa V1).
- Compute Sanitizer và synccheck cho S4 V2 nếu có sẵn.
- Timing harness S1–S2 của V0.
- Full pipeline V0/V1/V2, correctness, timing và speedup.
- Benchmark 12 case và khung báo cáo 5 phần của bản tối ưu.

> Chọn **Runtime → Change runtime type → T4 GPU** trong Google Colab.
> Cell setup clone nhánh `v2`: cần push nhánh chứa source V2 lên GitHub trước,
> hoặc đặt `USE_EXISTING_SOURCE = True` khi đã đưa source vào `REPO_DIR`.
> Notebook được chuẩn bị với sự hỗ trợ của OpenAI Codex; chưa có số đo thực tế.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = 'https://github.com/nganbee/panorama-stitching-project.git'
BRANCH = 'v2'
USE_EXISTING_SOURCE = False
REPO_DIR = Path('/content/panorama-stitching-project')

def run(command, check=True):
    print(f'$ {command}')
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')
    return result.returncode

if USE_EXISTING_SOURCE:
    if not (REPO_DIR / 'src/stage_implementations/v2_stage4.cu').is_file():
        raise RuntimeError('Chưa có source V2 tại REPO_DIR; hãy đưa source lên Colab trước.')
elif not REPO_DIR.exists():
    run(f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
else:
    run(f'git -C {REPO_DIR} fetch origin {BRANCH}')
    run(f'git -C {REPO_DIR} checkout {BRANCH}')
    run(f'git -C {REPO_DIR} pull --ff-only origin {BRANCH}')

os.chdir(REPO_DIR)
Path('build').mkdir(exist_ok=True)
Path('out').mkdir(exist_ok=True)
Path('reports').mkdir(exist_ok=True)
print('Working directory:', Path.cwd())

## 1. Kiểm tra GPU và CUDA

In [ ]:
run('nvidia-smi')
run('nvcc --version')
print('nvcc:', shutil.which('nvcc'))
print('compute-sanitizer:', shutil.which('compute-sanitizer'))

## 2. Build và chạy V0 CPU reference

V0 được dùng làm baseline để kiểm tra kết quả và timing.

In [ ]:
run('g++ -O2 -std=c++17 -Isrc/core_headers -Isrc/libraries '
    'src/integration/main.cpp src/integration/v0_pipeline.cpp '
    'src/stage_implementations/v0_stage12.cpp '
    'src/stage_implementations/v0_stage34.cpp '
    'src/stage_implementations/v0_stage5.cpp '
    'src/stage_implementations/v0_stage6.cpp '
    'src/libraries/stb_impl.cpp -o build/stitch')

In [ ]:
LEFT = 'dataset/location1/small_720p/1.jpg'
RIGHT = 'dataset/location1/small_720p/2.jpg'
V0_OUTPUT = 'out/panorama_v0_location1_720p.png'
V0_JSON = 'reports/v0_location1_720p.json'
run(f'build/stitch {LEFT} {RIGHT} {V0_OUTPUT} --version 0 --runs 5 --json {V0_JSON}')

In [ ]:
from IPython.display import Image, display
display(Image(filename=V0_OUTPUT))

## 3. Self-test S1–S2 CUDA

Test này so sánh grayscale, Gaussian blur, Harris response, NMS và top-K của V1 với V0.

In [ ]:
run('nvcc -O2 -std=c++17 --fmad=false -Isrc/core_headers '
    'tools/run_stage12_cuda_selftest.cpp '
    'src/stage_implementations/v0_stage12.cpp '
    'src/stage_implementations/v1_stage12.cu '
    '-o build/stage12_cuda_selftest')
run('build/stage12_cuda_selftest')

In [ ]:
if shutil.which('compute-sanitizer'):
    run('compute-sanitizer --error-exitcode 1 build/stage12_cuda_selftest')
else:
    print('compute-sanitizer is not available; skipped.')

## 4. Self-test S3–S4 CUDA

So sánh descriptor và matches với V0; kiểm tra cả S4 V1 và S4 V2.
Các ca V2 gồm tile 127/128/129, tile cuối không đủ thread, descriptor trùng,
tie-break qua biên tile, input rỗng và 3000 keypoint.


In [ ]:
run('nvcc -O2 -std=c++17 --fmad=false -Isrc/core_headers '
    'tools/run_stage34_v2_cuda_selftest.cpp '
    'src/stage_implementations/v0_stage34.cpp '
    'src/stage_implementations/v1_stage34.cu '
    'src/stage_implementations/v2_stage4.cu '
    '-o build/stage34_v2_selftest')
run('build/stage34_v2_selftest')
if shutil.which('compute-sanitizer'):
    run('compute-sanitizer --error-exitcode 1 build/stage34_v2_selftest')
    run('compute-sanitizer --tool synccheck --error-exitcode 1 build/stage34_v2_selftest')
else:
    print('compute-sanitizer is not available; memory/barrier checks NOT VERIFIED.')


### Self-test S5 CUDA

In [ ]:
run('nvcc -O2 -std=c++17 --fmad=false -Isrc/core_headers '
    'tools/test_v1_stage5.cu '
    'src/stage_implementations/v0_stage5.cpp '
    '-o build/stage5_cuda_selftest')
run('build/stage5_cuda_selftest')

In [ ]:
if shutil.which('compute-sanitizer'):
    run('compute-sanitizer --error-exitcode 1 build/stage5_cuda_selftest')
else:
    print('compute-sanitizer is not available; skipped.')

## 5. Self-test S6 CUDA

In [ ]:
run('nvcc -O2 -std=c++17 --fmad=false -Isrc/core_headers '
    'tools/test_v1_stage6.cpp '
    'src/stage_implementations/v0_stage6.cpp '
    'src/stage_implementations/v1_stage6.cu '
    '-o build/test_v1_stage6')
run('build/test_v1_stage6')

## 6. Timing harness S1–S2 của V0

Harness có một warm-up và trung bình tối thiểu năm lần chạy ở 720p, 1080p và 4K.

In [ ]:
run('g++ -O2 -std=c++17 -Isrc/core_headers -Isrc/libraries '
    'tools/timing_stage12.cpp '
    'src/stage_implementations/v0_stage12.cpp '
    'src/libraries/stb_impl.cpp '
    '-o build/timing_stage12')
run(f'build/timing_stage12 5 {LEFT} {RIGHT}')

## 7. Build và chạy full V2 pipeline

V2 giữ S1–S3, S5–S6 của V1; chỉ thay S4 bằng shared-memory tiling.
Mỗi block 128 thread xử lý 128 query, dùng chung tile 128 candidate (4 KB).
Giữ Hamming, top-2, ratio test, cross-check và thứ tự matches như V1.

Cùng binary chạy được `--version 0`, `1`, `2` để so sánh trong cùng session.


In [ ]:
BUILD_COMMAND = ('nvcc -O2 -std=c++17 --fmad=false -DPANORAMA_WITH_CUDA -DPANORAMA_WITH_V2 '
    '-Isrc/core_headers -Isrc/libraries '
    'src/integration/main.cpp src/integration/v0_pipeline.cpp '
    'src/stage_implementations/v0_stage12.cpp '
    'src/stage_implementations/v0_stage34.cpp '
    'src/stage_implementations/v0_stage5.cpp '
    'src/stage_implementations/v0_stage6.cpp '
    'src/stage_implementations/v1_stage12.cu '
    'src/stage_implementations/v1_stage34.cu '
    'src/stage_implementations/v1_stage5.cu '
    'src/stage_implementations/v1_stage6.cu '
    'src/stage_implementations/v2_stage4.cu '
    'src/cuda/v2_pipeline.cu '
    'src/cuda/cuda_pipeline.cu src/libraries/stb_impl.cpp '
    '-o build/stitch_v2')
run(BUILD_COMMAND)
run('build/stitch_v2 --list-versions')


In [ ]:
V1_OUTPUT = 'out/panorama_v1_location1_720p.png'
V1_JSON = 'reports/v1_location1_720p.json'
V2_OUTPUT = 'out/panorama_v2_location1_720p.png'
V2_JSON = 'reports/v2_location1_720p.json'
run(f'build/stitch_v2 {LEFT} {RIGHT} {V1_OUTPUT} --version 1 --runs 5 --json {V1_JSON}')
run(f'build/stitch_v2 {LEFT} {RIGHT} {V2_OUTPUT} --version 2 --runs 5 --json {V2_JSON}')

# Test tích hợp riêng: kiểm tra V0/V1/V2 và timing contract.
integration_build = BUILD_COMMAND.replace('src/integration/main.cpp', 'tools/test_v2_pipeline.cpp')
integration_build = integration_build.replace('-o build/stitch_v2', '-o build/test_v2_pipeline')
run(integration_build)
run('build/test_v2_pipeline')


In [ ]:
from PIL import Image as PILImage
import numpy as np
import json

v2 = np.asarray(PILImage.open(V2_OUTPUT).convert('RGB')).astype(np.int16)
with open(V2_JSON, encoding='utf-8') as f:
    v2_check = json.load(f)
for label, image_path, report_path, tolerance in [
    ('V0', V0_OUTPUT, V0_JSON, 1), ('V1', V1_OUTPUT, V1_JSON, 0)
]:
    reference = np.asarray(PILImage.open(image_path).convert('RGB')).astype(np.int16)
    print(label, 'shape:', reference.shape, '| V2 shape:', v2.shape)
    assert reference.shape == v2.shape, f'{label}/V2 panorama dimensions differ'
    diff = np.abs(reference - v2)
    print(label, 'vs V2 mean_abs_pixel_error:', float(diff.mean()))
    print(label, 'vs V2 max_abs_pixel_error:', int(diff.max()))
    assert diff.max() <= tolerance, f'{label}/V2 pixel mismatch'
    with open(report_path, encoding='utf-8') as f:
        ref = json.load(f)
    for key in ['keypoints_A', 'keypoints_B', 'matches', 'inliers']:
        assert ref['features'][key] == v2_check['features'][key], (label, key)
    assert np.allclose(ref['homography_B_to_A'], v2_check['homography_B_to_A'],
                       rtol=0, atol=1e-5 if label == 'V0' else 0)
    print(f'Correctness PASS: V2 vs {label}')
display(Image(filename=V2_OUTPUT))


## 8. So sánh timing V0 CPU, V1 CUDA và V2 CUDA

Đọc ba JSON report, giữ kiểu bảng và biểu đồ như notebook V1.
`V0/V2` cho speedup so với CPU; `V1/V2` đo hiệu quả của tiling.
Giá trị > 1 nghĩa là V2 nhanh hơn bản đối chiếu.

Stage V1/V2 là kernel time bằng CUDA events; H2D/D2H đo riêng.
End-to-end gồm allocation, launch, đồng bộ và truyền dữ liệu, loại đọc/ghi ảnh.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

reports = {}
for version, path in [(0, V0_JSON), (1, V1_JSON), (2, V2_JSON)]:
    with open(path, encoding='utf-8') as f:
        reports[version] = json.load(f)
stage_names = {
    'S1': 'S1: grayscale + blur', 'S2': 'S2: Harris + NMS',
    'S3': 'S3: descriptor', 'S4': 'S4: matching',
    'S5': 'S5: RANSAC', 'S6': 'S6: warp + blend',
}
rows = []
for stage, label in list(stage_names.items()) + [('end_to_end', 'End-to-end')]:
    values = [float(reports[v]['timings_ms']['end_to_end'] if stage == 'end_to_end'
                    else reports[v]['stage_timings_ms'][stage]) for v in [0, 1, 2]]
    a, b, c = values
    rows.append({'Stage': label, 'V0 CPU (ms)': a, 'V1 CUDA (ms)': b,
                 'V2 CUDA (ms)': c, 'Speedup V0/V2': a / c if c > 0 else float('nan'),
                 'Speedup V1/V2': b / c if c > 0 else float('nan')})
comparison = pd.DataFrame(rows)
display(comparison.style.format({col: '{:.3f}' for col in comparison.columns if col != 'Stage'}))
plot_df = comparison[comparison.Stage != 'End-to-end'].set_index('Stage')
ax = plot_df[['V0 CPU (ms)', 'V1 CUDA (ms)', 'V2 CUDA (ms)']].plot(
    kind='bar', figsize=(12, 5), logy=True)
ax.set_ylabel('Time (ms, logarithmic scale)')
ax.set_title('V0 CPU vs V1 CUDA vs V2 CUDA by stage')
ax.grid(axis='y')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

breakdown = []
for category in ['kernel_total', 'h2d', 'd2h', 'end_to_end']:
    breakdown.append({'Category': category,
                      'V1 (ms)': reports[1]['timings_ms'][category],
                      'V2 (ms)': reports[2]['timings_ms'][category]})
display(pd.DataFrame(breakdown))
for version in [1, 2]:
    times = reports[version]['timings_ms']
    kernel_sum = sum(reports[version]['stage_timings_ms'].values())
    assert abs(kernel_sum - times['kernel_total']) <= 1e-5 * max(1.0, kernel_sum)
    assert all(times[key] > 0 for key in ['kernel_total', 'h2d', 'd2h', 'end_to_end'])
print('Timing PASS: kernel totals agree; H2D/D2H and end-to-end are populated.')
v1_s4_share = reports[1]['stage_timings_ms']['S4'] / reports[1]['timings_ms']['kernel_total']
print(f'V1 S4 share of GPU kernel time: {v1_s4_share:.2%}')


### Cách diễn giải

- `Speedup V1/V2 > 1`: tiling giúp nhanh hơn V1; < 1: giữ kết quả và phân tích nguyên nhân.
- S4 giảm thời gian nhưng end-to-end ít đổi có thể do copy/allocation hoặc stage khác chiếm ưu thế.
- Tỷ trọng S4 ở trên tính trên kernel_total; xem cả H2D/D2H và end-to-end khi kết luận bottleneck.
- Chỉ S4 đổi; dao động thời gian ở stage khác không phải tối ưu mới của V2.
- V1 đã có cache GPU; shared-memory tiling thêm barrier nên không bảo đảm nhanh hơn mọi case.


## 9. Kiểm tra trạng thái full V2 pipeline


In [ ]:
for label, path in [
    ('V2 S4 source', 'src/stage_implementations/v2_stage4.cu'),
    ('V2 pipeline', 'src/cuda/v2_pipeline.cu'),
    ('Executable', 'build/stitch_v2'), ('V2 report', V2_JSON)
]:
    print(label, ':', path, '| exists:', Path(path).is_file())
    assert Path(path).is_file(), path


## 10. Kết quả mong đợi

Các cell cần lần lượt đạt:

```text
S1/S2 CUDA parity tests passed
S3/S4 CUDA parity tests passed
V2 tiled matching parity tests passed
PASS: S5 CUDA sample and V0 parity tests
Result: ALL PASS
version=1 ...
version=2 ...
PASS: full V1 ...
PASS: full V2 ...
Correctness PASS: V2 vs V0
Correctness PASS: V2 vs V1
Timing PASS: ...
```

Đây là output mong đợi, không phải kết quả đã đo. Nếu test/pipeline thất bại,
giữ log đầu tiên, thông tin GPU/CUDA và case ảnh để debug.


## 11. Benchmark đủ 12 case

3 location chuẩn + 1 challenge, mỗi cặp có 720p/1080p/4K.
Chạy V0/V1/V2 trong cùng session/GPU; mỗi case một warm-up và năm lần đo.
Lưu environment, JSON, log, panorama và bảng tổng hợp. Case lỗi được ghi rõ,
không tính speedup cho output sai. Script báo lỗi khi chưa đạt đủ các case.


In [ ]:
import sys
import shlex
benchmark_status = run(f'{shlex.quote(sys.executable)} tools/benchmark_v2.py', check=False)
summary_path = Path('reports/v2_benchmark/summary.csv')
assert summary_path.is_file(), 'Chưa tạo summary.csv; xem log benchmark phía trên.'
summary = pd.read_csv(summary_path)
display(summary)
valid = summary[summary.status == 'pass'].set_index('case')
columns = ['v1_S4_kernel_share', 'v0_over_v2_match_speedup',
           'v1_over_v2_match_speedup', 'v0_over_v2_end_to_end_speedup',
           'v1_over_v2_end_to_end_speedup']
display(valid.reindex(columns=columns))
if len(valid):
    ax = valid.reindex(columns=['v1_over_v2_match_speedup',
                               'v1_over_v2_end_to_end_speedup']).plot.bar(figsize=(14, 5))
    ax.axhline(1, color='black', linestyle='--')
    ax.set_ylabel('V1 time / V2 time')
    plt.tight_layout()
    plt.show()
assert benchmark_status == 0, 'Benchmark chưa đạt; xem status và log từng case.'


## 12. Báo cáo V2 — điền từ output thực tế

### a. Bottleneck
Đưa timing/profiling V1, tỷ trọng S4 và lý do chọn S4 để tối ưu.

### b. Idea
Tile 128 candidate vào shared memory để tối đa 128 query tái sử dụng;
giảm đọc global memory, đổi lại có chi phí đồng bộ.

### c. Implementation
Trình bày `v2_stage4.cu`, thread/block mapping, hai barrier, tile cuối,
hai chiều matching và các quy tắc ratio/cross-check/tie-break giữ nguyên.

### d. Result
Dùng output kiểm chứng, bảng 12 case và biểu đồ ở trên: correctness,
S4/kernel/copy/end-to-end, speedup so với V0 và V1.

### e. Analysis
Giải thích case nhanh/chậm, mức thay đổi S4 so với end-to-end, cache và
barrier overhead. Không suy ra kết quả trước khi chạy.

Lưu toàn bộ output cells sau Run all; mọi số liệu trong báo cáo/slides phải
truy được về output notebook. Các nguồn/ý tưởng bên ngoài cần trích tại nơi sử dụng.
